In [0]:
"""
Module: Pipeline Configuration
Description: Initializes dynamic Databricks widgets for environment 
             and catalog parameterization to support CI/CD workflows.
"""
dbutils.widgets.text("catalog_name", "nyc_taxi_dev", "Target Catalog")
dbutils.widgets.text("env", "dev", "Environment")

CATALOG = dbutils.widgets.get("catalog_name")
ENV = dbutils.widgets.get("env")

print(f"✅ Pipeline Initialized. Env: {ENV} | Target Catalog: {CATALOG}")

In [0]:
    """
Module: Task A1 - Ingest Taxi Zones (Dimension)
Description: Extracts the static TLC Taxi Zone lookup table.
             Enforces pure StringType across all columns.
Target: {CATALOG}.bronze.raw_zones
"""
import pandas as pd
from pyspark.sql.types import StringType
from pyspark.sql.functions import col

target_zones = f"{CATALOG}.bronze.raw_zones"
zone_url = "https://d37ci6vzurychx.cloudfront.net/misc/taxi+_zone_lookup.csv"

# Load via Pandas (as strings), convert to Spark, and explicitly cast again
pdf = pd.read_csv(zone_url, dtype=str)
df_zones = spark.createDataFrame(pdf)

for c in df_zones.columns:
  df_zones = df_zones.withColumn(c, col(c).cast(StringType()))

# Execute full overwrite
df_zones.write.mode("overwrite").format("delta").saveAsTable(target_zones)

print(f"✅ Task A1: Dimension data loaded as strict StringType to {target_zones}")

In [0]:
"""
Module: Task A2 - Volume-Backed Parquet Ingestion (Fact)
Description: Stateful pipeline that downloads massive Parquet files into a 
             UC Volume, reads them, and explicitly casts ALL columns to StringType.
Target: {CATALOG}.bronze.raw_trips
Volume: /Volumes/{CATALOG}/bronze/raw_landing/
"""
import urllib.request
import os
from pyspark.sql.types import StringType
from pyspark.sql.functions import col, md5, concat_ws, current_timestamp
from delta.tables import DeltaTable
from dateutil.relativedelta import relativedelta

target_trips = f"{CATALOG}.bronze.raw_trips"
volume_dir = f"/Volumes/{CATALOG}/bronze/raw_landing"

# 1. Determine High Watermark
if not spark.catalog.tableExists(target_trips):
    next_year, next_month = 2014, 1
else:
    max_date = spark.sql(f"SELECT MAX(tpep_pickup_datetime) as md FROM {target_trips}").collect()[0]['md']
    # Safely calculate next month even though the date is stored as a string
    next_date = spark.sql(f"SELECT CAST('{max_date}' AS DATE) + INTERVAL 1 MONTH as nd").collect()[0]['nd']
    next_year, next_month = next_date.year, next_date.month

# 2. Stage to UC Volume
parquet_filename = f"yellow_tripdata_{next_year}-{next_month:02d}.parquet"
parquet_url = f"https://d37ci6vzurychx.cloudfront.net/trip-data/{parquet_filename}"
volume_file_path = f"{volume_dir}/{parquet_filename}"

if not os.path.exists(volume_file_path):
    print(f"⏳ Downloading {parquet_filename} to UC Volume... (takes 1-2 mins)")
    req = urllib.request.Request(parquet_url, headers={'User-Agent': 'Mozilla/5.0'})
    with urllib.request.urlopen(req) as response, open(volume_file_path, 'wb') as out_file:
        out_file.write(response.read())

# 3. Read Parquet and Enforce StringType on ALL columns
df_raw = spark.read.parquet(volume_file_path)
df_string = df_raw.select([col(c).cast(StringType()).alias(c) for c in df_raw.columns])

# 4. Generate Surrogate Key & Ingestion Timestamp
df_hashed = df_string.withColumn("trip_hash_id", 
    md5(concat_ws("_", col("VendorID"), col("tpep_pickup_datetime"), col("trip_distance")))
).withColumn("ingested_at", current_timestamp())

# 5. Execute Idempotent Write (Merge)
if not spark.catalog.tableExists(target_trips):
    df_hashed.write.format("delta").saveAsTable(target_trips)
    print(f"🚀 Task A2: Initial Bulk Load ({next_year}-{next_month:02d}) created.")
else:
    DeltaTable.forName(spark, target_trips).alias("t").merge(
        df_hashed.alias("s"), "t.trip_hash_id = s.trip_hash_id"
    ).whenNotMatchedInsertAll().execute()
    print(f"🔄 Task A2: Bulk Parquet data ({next_year}-{next_month:02d}) merged successfully.")

In [0]:
"""
Module: Task B - Dynamic Weather Ingestion (Visual Crossing)
Description: Independent stateful pipeline using the Visual Crossing API.
             Fetches the next month of 2014 weather data as a CSV string 
             to bypass JSON parsing limits and enforces StringType.
Target: {CATALOG}.bronze.raw_weather
"""
import requests
import io
import pandas as pd
from pyspark.sql.types import StringType
from pyspark.sql.functions import col, current_timestamp
from delta.tables import DeltaTable
from dateutil.relativedelta import relativedelta

# --- CONFIGURATION ---
VC_API_KEY = "QWPUUJNGUQDG278ESUDZTZ7UP" # <--- Paste your key here
target_weather = f"{CATALOG}.bronze.raw_weather"

# 1. Determine High Watermark (Statefulness)
if not spark.catalog.tableExists(target_weather):
    next_year, next_month = 2014, 1
else:
    max_date = spark.sql(f"SELECT MAX(date) as md FROM {target_weather}").collect()[0]['md']
    next_date = spark.sql(f"SELECT CAST('{max_date}' AS DATE) + INTERVAL 1 DAY as nd").collect()[0]['nd']
    next_year, next_month = next_date.year, next_date.month

# 2. Parameterize API Request
start_date = f"{next_year}-{next_month:02d}-01"
end_date = spark.sql(f"SELECT (CAST('{start_date}' AS DATE) + INTERVAL 1 MONTH - INTERVAL 1 DAY)").collect()[0][0]

print(f"🌦️ Fetching Weather Metrics: TempMax, TempMin, Temp, Precip for {start_date} to {end_date}")

# Added tempmax, tempmin, and temp to the 'elements' parameter
url = f"https://weather.visualcrossing.com/VisualCrossingWebServices/rest/services/timeline/40.71%2C-74.00/{start_date}/{end_date}?unitGroup=metric&elements=datetime%2Ctempmax%2Ctempmin%2Ctemp%2Cprecip&include=days&key={VC_API_KEY}&contentType=csv"

# 3. Fetch and Process Data
response = requests.get(url)

if response.status_code == 200:
    # Read CSV response into Pandas as strings to preserve raw values
    pdf = pd.read_csv(io.StringIO(response.text), dtype=str)
    
    # Map the Visual Crossing column names to our consistent naming convention
    # datetime -> date, precip -> rain_mm, temp -> avg_temp
    df_weather = spark.createDataFrame(pdf) \
                      .withColumnRenamed("datetime", "date") \
                      .withColumnRenamed("precip", "rain_mm") \
                      .withColumnRenamed("temp", "avg_temp")
    
    # Strictly enforce StringType on all columns (The Raw String Pattern)
    for c in df_weather.columns:
        df_weather = df_weather.withColumn(c, col(c).cast(StringType()))
        
    df_weather = df_weather.withColumn("ingested_at", current_timestamp())

    # 4. Execute Idempotent Write (Merge)
    if not spark.catalog.tableExists(target_weather):
        df_weather.write.format("delta").saveAsTable(target_weather)
        print(f"🚀 Task B: Initial Weather Load for {next_year}-{next_month:02d} created with enhanced metrics.")
    else:
        dt = DeltaTable.forName(spark, target_weather)
        dt.alias("target").merge(
            df_weather.alias("source"),
            "target.date = source.date"
        ).whenNotMatchedInsertAll().execute()
        print(f"🔄 Task B: Weather metrics for {next_year}-{next_month:02d} merged successfully.")
else:
    print(f"❌ API Error {response.status_code}: {response.text}")